# Вебинар 3: Деревья решений, Random Forest и Boosting в финтех задачах

## 📊 Датасет
**Название:** [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
**Описание:** Анонимизированные транзакции по кредитным картам европейских держателей карт (сентябрь 2013).  
**Размер:** 284 807 транзакций, 31 признак (V1-V28 — PCA-признаки, Time, Amount, Class).  
**Задача:** Обнаружение мошеннических транзакций (дисбаланс классов ~0.17%).  
**Бизнес-применение:** Anti-fraud системы реального времени.

## 🎯 Цели ноутбука
1. Сравнить Decision Tree, Random Forest, XGBoost, LightGBM на дисбалансных данных
2. Построить XGBoost с PR-AUC > 0.7
3. Настроить порог классификации
4. Построить Stacking ансамбль
5. Визуализировать SHAP values

## 1. Импорт и загрузка

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, KFold, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    confusion_matrix, precision_recall_curve, roc_curve
)

import xgboost as xgb
import lightgbm as lgb

df = pd.read_csv('/kaggle/input/creditcardfraud/creditcard.csv')
print(f"Размер: {df.shape}")
print(f"\nРаспределение классов:")
print(df['Class'].value_counts())
print(f"\nДоля мошенничества: {df['Class'].mean():.4%}")

In [ ]:
df.head()

In [ ]:
print("Статистика по суммам транзакций:")
print(df['Amount'].describe())
print(f"\nМаксимальная сумма мошенничества: {df[df['Class']==1]['Amount'].max():.2f}")
print(f"Средняя сумма мошенничества: {df[df['Class']==1]['Amount'].mean():.2f}")
print(f"Средняя сумма обычной: {df[df['Class']==0]['Amount'].mean():.2f}")

## 2. Подготовка данных

In [ ]:
# Нормализуем Time и Amount
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])

feature_cols = [c for c in df.columns if c not in ['Class', 'Time', 'Amount']]
X = df[feature_cols].values
y = df['Class'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, fraud: {y_train.sum()} ({y_train.mean():.4%})")
print(f"Test:  {X_test.shape}, fraud: {y_test.sum()} ({y_test.mean():.4%})")

## 3. Сравнение моделей: дерево vs ансамбли

In [ ]:
# === Модели для сравнения ===
models = {
    'Decision Tree (depth=5)': DecisionTreeClassifier(max_depth=5, random_state=42),
    'Decision Tree (depth=15)': DecisionTreeClassifier(max_depth=15, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42),
    'XGBoost': xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                   scale_pos_weight=200, random_state=42, eval_metric='aucpr', n_jobs=-1),
    'LightGBM': lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                     random_state=42, verbose=-1, n_jobs=-1),
}

results = {}
print("Сравнение моделей на дисбалансных данных:")
print("=" * 80)
print(f"{'Model':<30} {'ROC-AUC':>10} {'PR-AUC':>10} {'F1':>10}")
print("=" * 80)

for name, model in models.items():
    model.fit(X_train, y_train)
    y_proba = model.predict_proba(X_test)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)
    from sklearn.metrics import f1_score
    f1 = f1_score(y_test, y_pred)

    results[name] = {'model': model, 'roc_auc': roc_auc, 'pr_auc': pr_auc, 'f1': f1, 'proba': y_proba}
    print(f"{name:<30} {roc_auc:>10.4f} {pr_auc:>10.4f} {f1:>10.4f}")

## 4. Лучший XGBoost: тюнинг порога

In [ ]:
# Лучшая модель
best_model_name = max(results, key=lambda k: results[k]['pr_auc'])
best_model = results[best_model_name]['model']
y_proba = results[best_model_name]['proba']

print(f"Лучшая модель: {best_model_name}")
print(f"PR-AUC: {results[best_model_name]['pr_auc']:.4f}")

In [ ]:
# === Анализ precision-recall ===
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

# Находим порог с максимальным F1
f1_scores = 2 * (precision * recall) / (precision + recall + 1e-8)
best_threshold_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_threshold_idx] if best_threshold_idx < len(thresholds) else 0.5

print(f"Лучший порог (F1): {best_threshold:.4f}")
print(f"  Precision: {precision[best_threshold_idx]:.4f}")
print(f"  Recall: {recall[best_threshold_idx]:.4f}")
print(f"  F1: {f1_scores[best_threshold_idx]:.4f}")

# Бизнес-ориентированный порог: Precision >= 0.9
valid_idx = np.where(precision >= 0.9)[0]
if len(valid_idx) > 0:
    business_idx = valid_idx[-1]  # Максимальный recall при Precision >= 0.9
    business_threshold = thresholds[business_idx] if business_idx < len(thresholds) else 0.5
    print(f"\nБизнес-порог (Precision >= 0.9): {business_threshold:.4f}")
    print(f"  Precision: {precision[business_idx]:.4f}")
    print(f"  Recall: {recall[business_idx]:.4f}")

In [ ]:
# Визуализация Precision-Recall
plt.figure(figsize=(10, 6))
plt.plot(recall, precision, linewidth=2, label=f'PR-кривая (AUC={results[best_model_name]["pr_auc"]:.4f})')
plt.scatter(recall[best_threshold_idx], precision[best_threshold_idx], color='red', s=200, zorder=5,
            label=f'F1-optimum (t={best_threshold:.3f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall кривая')
plt.legend()
plt.grid(alpha=0.3)
plt.xlim([0, 1.05])
plt.ylim([0, 1.05])
plt.show()

In [ ]:
# === Оценка с лучшим порогом ===
y_pred_optimized = (y_proba >= best_threshold).astype(int)
print(f"\nClassification Report (порог={best_threshold:.4f}):")
print(classification_report(y_test, y_pred_optimized, target_names=['Normal', 'Fraud']))

cm = confusion_matrix(y_test, y_pred_optimized)
print(f"\nConfusion Matrix:")
print(f"  TN={cm[0,0]:>6}  FP={cm[0,1]:>4}")
print(f"  FN={cm[1,0]:>4}  TP={cm[1,1]:>4}")

# Бизнес-стоимость
cost_fn = 1000  # Мошенничество пропущено: средний ущерб $1000
cost_fp = 50    # Ложная тревога: проверка операции $50
total_cost = cm[1,0] * cost_fn + cm[0,1] * cost_fp
print(f"\nСтоимость ошибок: ${total_cost:,}")
print(f"  FN (пропущено мошенничество): ${cm[1,0] * cost_fn:,}")
print(f"  FP (ложные тревоги): ${cm[0,1] * cost_fp:,}")

## 5. Stacking ансамбль

In [ ]:
# === Stacking: 3 base models + meta-learner ===
base_models = [
    ('rf', RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)),
    ('xgb', xgb.XGBClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                               scale_pos_weight=200, random_state=42, eval_metric='aucpr', n_jobs=-1)),
    ('lgb', lgb.LGBMClassifier(n_estimators=200, max_depth=5, learning_rate=0.1,
                                 random_state=42, verbose=-1, n_jobs=-1)),
]

stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=1000, random_state=42),
    cv=5,
    n_jobs=-1,
    passthrough=False
)

stacking.fit(X_train, y_train)
y_proba_stack = stacking.predict_proba(X_test)[:, 1]

stack_pr_auc = average_precision_score(y_test, y_proba_stack)
stack_roc_auc = roc_auc_score(y_test, y_proba_stack)
print(f"Stacking PR-AUC:  {stack_pr_auc:.4f}")
print(f"Stacking ROC-AUC: {stack_roc_auc:.4f}")
print(f"\nЛучшая одиночная: {results[best_model_name]['pr_auc']:.4f}")

In [ ]:
# Сравнение ROC-кривых
plt.figure(figsize=(10, 7))
for name, data in results.items():
    fpr, tpr, _ = roc_curve(y_test, data['proba'])
    plt.plot(fpr, tpr, linewidth=2, alpha=0.7, label=f"{name} (PR-AUC={data['pr_auc']:.3f})")

fpr, tpr, _ = roc_curve(y_test, y_proba_stack)
plt.plot(fpr, tpr, linewidth=3, color='red', label=f'STACKING (PR-AUC={stack_pr_auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', alpha=0.5)
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Сравнение ROC-кривых (Fraud Detection)')
plt.legend(loc='lower right', fontsize=9)
plt.grid(alpha=0.3)
plt.show()

## 6. SHAP values

In [ ]:
try:
    import shap
except ImportError:
    !pip install shap -q
    import shap

In [ ]:
# Используем лучшую XGBoost модель
xgb_best = results['XGBoost']['model']
explainer = shap.TreeExplainer(xgb_best)

sample_idx = np.random.choice(len(X_test), 1000, replace=False)
X_sample = X_test[sample_idx]
shap_values = explainer.shap_values(X_sample)

# Summary plot
feature_names = [f'V{i}' for i in range(1, 29)] + ['Amount_scaled', 'Time_scaled']
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, show=False)
plt.title('SHAP values (топ-10 признаков)')
plt.tight_layout()
plt.show()

In [ ]:
# === Локальная интерпретация: топ-мошеннические транзакции ===
top_fraud_idx = np.argsort(y_proba)[-3:]

fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for i, idx in enumerate(top_fraud_idx):
    if idx in sample_idx:
        sample_pos = list(sample_idx).index(idx)
        shap.force_plot(explainer.expected_value, shap_values[sample_pos], X_test[idx],
                        feature_names=feature_names, matplotlib=True, show=False, ax=axes[i])
    axes[i].set_title(f'Транзакция {idx}: P(fraud)={y_proba[idx]:.3f}')
plt.tight_layout()
plt.show()

## 📋 Выводы

1. **Лучшие модели:** XGBoost и LightGBM значительно превосходят одно дерево на дисбалансных данных
2. **PR-AUC > 0.7** достигнут за счёт `scale_pos_weight` и градиентного бустинга
3. **Бизнес-порог** зависит от соотношения стоимости FN/FP
4. **Stacking** может дать дополнительный прирост к качеству
5. **SHAP** показывает, что признаки V14, V12, V10 — главные индикаторы мошенничества